<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Aprender Guitarra Después de Tocar Piano 🎸🎹
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/01_Transfer_Learning_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 08 (Transfer Learning y *Fine-Tuning*)](../01_Transfer_Learning_y_Fine_Tuning.ipynb). La idea: **no empezar de cero** cada vez. Si ya sabes tocar el piano, aprender guitarra te cuesta **menos lecciones**: ya tienes **oído musical** y **sentido del ritmo**. Eso se llama ***transfer learning*** (transferir lo aprendido). Lo veremos con un mini-experimento en *NumPy* y *scikit-learn* (sin redes neuronales).

Al terminar podrás explicar:
1. Qué se **reutiliza** (lo general: el oído) y qué se **vuelve a aprender** (lo específico: los dedos).
2. Por qué la reutilización **ayuda mucho cuando tienes pocos ejemplos** y **ayuda menos** cuando tienes muchos.
3. Por qué **no siempre ayuda**: si la tarea nueva es de otro tipo, lo aprendido puede **no servir**.

> 🔗 Si luego quieres ver la versión con redes neuronales y más detalle: [cuaderno estándar](../01_Transfer_Learning_y_Fine_Tuning.ipynb). Antes de esto, la [introducción del capítulo](00_Introduccion_Cap3_Dummies.ipynb).

---
## 1. El oído musical que se reutiliza 🎼

| En la música | En machine learning | ¿Se reutiliza? |
|---|---|---|
| **Oído y ritmo** (sirven para cualquier instrumento) | Los **rasgos generales** que aprendió el modelo (formas, bordes, patrones) | ✅ **Sí**: es lo valioso |
| **Dónde poner los dedos en el piano** | La **parte final**, específica de la primera tarea | ❌ Se vuelve a aprender para la nueva |
| **Pocas lecciones de guitarra** | **Pocos ejemplos etiquetados** en la tarea nueva | Aquí es donde más se nota la ayuda |

Para verlo, creamos un **mundo de puntos en un plano**. **Tarea 1 (la que ya dominas, con *muchísimos* ejemplos):** decidir si un punto está **dentro de un círculo pequeño**. Para lograrlo, el modelo descubre una **cualidad útil de cada punto**: *«qué tan lejos está del centro»* (su «oído musical»). Guardamos esa cualidad como un **medidor**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(1)

def puntos(n):
    return rng.uniform(-2.5, 2.5, (n, 2))

# --- TAREA 1 (fuente): muchisimos ejemplos; la regla es "dentro de un circulo de radio 1" ---
X1 = puntos(3000)
y1 = (np.hypot(X1[:, 0], X1[:, 1]) < 1.0).astype(int)
modelo1 = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), LogisticRegression(C=100, max_iter=5000)).fit(X1, y1)

def medidor(X):
    """El 'oido musical' aprendido: un solo numero por punto que, en esencia, mide que tan lejos esta del centro."""
    return modelo1.decision_function(X)[:, None]

# Veamos que mide el medidor: deberia bajar a medida que el punto se aleja del centro
Xg = puntos(2000)
r = np.hypot(Xg[:, 0], Xg[:, 1])
correlacion = np.corrcoef(r, medidor(Xg)[:, 0])[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sc = axes[0].scatter(Xg[:, 0], Xg[:, 1], c=medidor(Xg)[:, 0], cmap="coolwarm", s=8)
axes[0].set_title("El 'medidor' aprendido en la Tarea 1", fontweight="bold", fontsize=10); axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); plt.colorbar(sc, ax=axes[0], label="valor del medidor")
axes[1].scatter(r, medidor(Xg)[:, 0], s=6, color="#2563eb", alpha=0.5)
axes[1].set_xlabel("Distancia real al centro"); axes[1].set_ylabel("Valor del medidor"); axes[1].set_title(f"Mide (casi) la distancia al centro (correlación {correlacion:.2f})", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Correlación entre el medidor y la distancia real al centro: {correlacion:.2f}  (el modelo descubrió una cualidad útil sin que nadie se la explicara).")
assert abs(correlacion) > 0.9

---
---
**Lección:** al resolver la Tarea 1 el modelo **descubrió** una cualidad general: «qué tan lejos del centro está un punto». Esa cualidad **no es solo para la Tarea 1**: sirve para **cualquier** pregunta sobre círculos.

## 2. Tarea nueva con pocos ejemplos: ¡a reutilizar! 🔁

**Tarea 2 (la nueva):** decidir si un punto está **dentro de un círculo más grande** (radio 1.7). Solo tenemos **unos pocos ejemplos etiquetados** (4, 8, 16, 32 o 128). Comparamos **tres estudiantes**, cada uno con **50 sorteos distintos** de ejemplos:

- 🔴 **El que usa una regla recta** (la más sencilla; no puede hacer círculos).
- 🟠 **El que empieza de cero con curvas** (aprende todo con esos pocos ejemplos).
- 🟢 **El que reutiliza el medidor** (solo aprende **dónde poner el corte**: la parte «dedos de la guitarra»).

In [ ]:
Xtest = puntos(5000)

def etiquetas_circulo(X):
    return (np.hypot(X[:, 0], X[:, 1]) < 1.7).astype(int)

def sorteo_con_ambas_clases(n, fy):
    while True:
        X = puntos(n); y = fy(X)
        if 0 < y.sum() < n:
            return X, y

def comparar(fy, tamanos, sorteos=50):
    resultados = {"Regla recta": [], "Curvas desde cero": [], "Reutiliza el medidor": []}
    yt = fy(Xtest)
    for n in tamanos:
        acc = {k: [] for k in resultados}
        for _ in range(sorteos):
            X, y = sorteo_con_ambas_clases(n, fy)
            acc["Regla recta"].append(LogisticRegression(C=100, max_iter=2000).fit(X, y).score(Xtest, yt))
            m = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), LogisticRegression(C=100, max_iter=5000)).fit(X, y)
            acc["Curvas desde cero"].append(m.score(Xtest, yt))
            acc["Reutiliza el medidor"].append(LogisticRegression(C=100, max_iter=2000).fit(medidor(X), y).score(medidor(Xtest), yt))
        for k in resultados:
            resultados[k].append(np.mean(acc[k]))
    return resultados

tamanos = [4, 8, 16, 32, 128]
res2 = comparar(etiquetas_circulo, tamanos)

fig, ax = plt.subplots(figsize=(8.8, 4.6))
for (nombre, v), c in zip(res2.items(), ["#dc2626", "#f59e0b", "#16a34a"]):
    ax.plot(tamanos, v, "o-", color=c, lw=2, label=nombre)
ax.set_xscale("log"); ax.set_xticks(tamanos); ax.set_xticklabels(tamanos); ax.minorticks_off(); ax.set_ylim(0.45, 1.02)
ax.set_xlabel("Ejemplos etiquetados de la Tarea 2"); ax.set_ylabel("Aciertos (promedio de 50 sorteos)"); ax.legend(fontsize=8)
ax.set_title("Con pocos ejemplos, reutilizar lo aprendido gana por mucho", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
for i, n in enumerate(tamanos):
    print(f"Con {n:>3} ejemplos: recta {res2['Regla recta'][i]:.0%} | curvas desde cero {res2['Curvas desde cero'][i]:.0%} | reutiliza el medidor {res2['Reutiliza el medidor'][i]:.0%}")
assert res2["Reutiliza el medidor"][0] > res2["Curvas desde cero"][0] + 0.15          # con 4 ejemplos, la ventaja es enorme
assert res2["Reutiliza el medidor"][-1] - res2["Curvas desde cero"][-1] < 0.05         # con 128 ejemplos casi no hay diferencia

---
---
**Lección:** con **4 ejemplos**, el que reutiliza el medidor acierta **~90 %**, mientras el que empieza de cero con curvas acierta **~65 %**; con **128 ejemplos** la diferencia casi desaparece. Igual que con la guitarra: **la ayuda del piano se nota sobre todo en las primeras lecciones**.

## 3. Cuando lo aprendido no sirve: la franja vertical ⛔

Ahora la **Tarea 3** es de **otro tipo**: decidir si un punto está en una **franja vertical** (|x| < 0.8). Esa pregunta **no tiene nada que ver** con la distancia al centro: el «oído» de los círculos **no ayuda**; incluso puede **estorbar**, porque obliga al estudiante a mirar solo ese medidor. Es como querer aprender **natación** reutilizando tu oído musical.

In [ ]:
def etiquetas_franja(X):
    return (np.abs(X[:, 0]) < 0.8).astype(int)

res3 = comparar(etiquetas_franja, tamanos)
fig, ax = plt.subplots(figsize=(8.8, 4.6))
for (nombre, v), c in zip(res3.items(), ["#dc2626", "#f59e0b", "#16a34a"]):
    ax.plot(tamanos, v, "o-", color=c, lw=2, label=nombre)
ax.set_xscale("log"); ax.set_xticks(tamanos); ax.set_xticklabels(tamanos); ax.minorticks_off(); ax.set_ylim(0.45, 1.02)
ax.set_xlabel("Ejemplos etiquetados de la Tarea 3"); ax.set_ylabel("Aciertos (promedio de 50 sorteos)"); ax.legend(fontsize=8)
ax.set_title("Si la tarea nueva es de otro tipo, lo reutilizado se queda corto", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
for i, n in enumerate(tamanos):
    print(f"Con {n:>3} ejemplos: recta {res3['Regla recta'][i]:.0%} | curvas desde cero {res3['Curvas desde cero'][i]:.0%} | reutiliza el medidor {res3['Reutiliza el medidor'][i]:.0%}")
assert res3["Curvas desde cero"][-1] > res3["Reutiliza el medidor"][-1] + 0.1           # con suficientes datos, aprender desde cero gana claramente

---
---
**Lección:** con **muchos ejemplos**, empezar **de cero** gana (≈ 98 %) frente a reutilizar el medidor equivocado (≈ 76 %): **lo reutilizado limita**. Reutilizar es una **apuesta** que solo sale bien si las dos tareas **se parecen**. Antes de adoptarla, **compárala siempre** con empezar de cero.

## Resumen en una frase 🎯

> **Reutilizar lo que otro modelo ya aprendió es como llegar a la guitarra sabiendo piano: te ahorra muchísimo cuando tienes pocas lecciones y las dos cosas se parecen, y ayuda poco (o estorba) cuando tienes muchas lecciones o el instrumento es muy distinto.**

### Lo que aprendiste hoy:
- ✅ **Se reutiliza lo general** (el oído, el «medidor») y **se vuelve a aprender lo específico** (los dedos, el corte).
- ✅ **Con muy pocos ejemplos** el que reutiliza gana por mucho (≈ 90 % frente a ≈ 65 % con 4 ejemplos); **con muchos**, la ventaja se acaba.
- ✅ **No siempre ayuda**: si la tarea nueva es de otro tipo (la franja), lo reutilizado **se queda corto**. **Compara siempre** con empezar de cero.
- ✅ Los modelos gigantes que se usan hoy (para imágenes o para texto) funcionan igual: **estudiaron muchísimo** y luego se **ajustan un poquito** para cada tarea.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../01_Transfer_Learning_y_Fine_Tuning.ipynb) hace el experimento con redes neuronales (dígitos 0–4 → 5–9), mide cuántas capas conviene reutilizar y cómo **afinar con delicadeza**; y luego, [cómo explicar las decisiones de un modelo](02_Explicabilidad_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Una tarea parecida: el círculo de radio 1.2

Reutilizar ayuda cuando la tarea nueva **se parece** a la que ya sabes. Probemos con una **Tarea 4**: decidir si un punto está dentro de un círculo de **radio 1.2** (parecida a la Tarea 1, igual que el círculo de 1.7). Usa la función `comparar` (que ya prueba a los tres estudiantes) y la lista `tamanos`:

1. Escribe la función `etiquetas_radio12(X)` que devuelva `1` cuando el punto está a menos de `1.2` del centro (usa `np.hypot`, como en `etiquetas_circulo`).
2. Llama a `comparar(etiquetas_radio12, [4, 16], sorteos=20)` y guarda el resultado en `res4` (con 20 sorteos para que sea rápido).
3. Imprime cuánto acierta cada estudiante con 4 y con 16 ejemplos.
4. Comprueba con `assert` que con **4 ejemplos** el que reutiliza el medidor le gana a «curvas desde cero» por más de 20 puntos, y que con 16 no queda por debajo.

In [ ]:
# Escribe tu código aquí

# def etiquetas_radio12(X):
#     return (np.hypot(X[:, 0], X[:, 1]) < ...).astype(int)
#
# res4 = comparar(etiquetas_radio12, [4, 16], sorteos=20)
# for i, n in enumerate([4, 16]):
#     print(n, res4["Regla recta"][i], res4["Curvas desde cero"][i], res4["Reutiliza el medidor"][i])


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
def etiquetas_radio12(X):
    return (np.hypot(X[:, 0], X[:, 1]) < 1.2).astype(int)      # 1 = dentro del circulo de radio 1.2

res4 = comparar(etiquetas_radio12, [4, 16], sorteos=20)
for i, n in enumerate([4, 16]):
    print(f"Con {n:>2} ejemplos: recta {res4['Regla recta'][i]:.0%} | curvas desde cero {res4['Curvas desde cero'][i]:.0%} | reutiliza el medidor {res4['Reutiliza el medidor'][i]:.0%}")

assert res4["Reutiliza el medidor"][0] > res4["Curvas desde cero"][0] + 0.2
assert res4["Reutiliza el medidor"][1] >= res4["Curvas desde cero"][1]
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué parte de lo aprendido con el piano reutilizas al aprender guitarra y qué parte tienes que aprender de nuevo? Da otro ejemplo de tu vida (idiomas, deportes, cocina).
2. En el experimento, ¿por qué reutilizar el «medidor» ayudó tanto con 4 ejemplos y casi nada con 128? ¿Qué le dirías a alguien que tiene un millón de ejemplos de la tarea nueva?
3. ¿Por qué el medidor de círculos **no sirvió** para la franja? ¿Cómo te darías cuenta, antes de gastar meses, de que la tarea que tienes **no se parece** a la que ya sabe el modelo?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>